In [1]:
import pandas as pd

df = pd.read_parquet(
    "../data/raw/complaint_embeddings.parquet"
)

print(df.head())

           id                                           document  \
0  14069121_0  a card was opened under my name by a fraudster...   
1  14061897_0  i made the mistake of using my wellsfargo debi...   
2  14061897_1  and got a letter stating my dispute was reject...   
3  14047085_0  dear cfpb, i have a secured credit card with c...   
4  14047085_1  y confirmation whatsoever to report to the pol...   

                                           embedding  \
0  [-0.04277738183736801, 0.025624370202422142, -...   
1  [-0.05458317697048187, 0.10340359061956406, 0....   
2  [-0.03491289168596268, 0.059216588735580444, 0...   
3  [-0.010181158781051636, 0.02354264445602894, -...   
4  [-0.017308838665485382, -0.007177562452852726,...   

                                            metadata  
0  {'chunk_index': 0, 'company': 'CITIBANK, N.A.'...  
1  {'chunk_index': 0, 'company': 'WELLS FARGO & C...  
2  {'chunk_index': 1, 'company': 'WELLS FARGO & C...  
3  {'chunk_index': 0, 'company': '

In [7]:
print(df.iloc[0]["metadata"])

{'chunk_index': 0, 'company': 'CITIBANK, N.A.', 'complaint_id': '14069121', 'date_received': '2025-06-13', 'issue': 'Getting a credit card', 'product': 'Credit card', 'product_category': 'Credit Card', 'state': 'TX', 'sub_issue': 'Card opened without my consent or knowledge', 'total_chunks': 1}


In [15]:
%load_ext autoreload
%autoreload 2
import pyarrow as pa
import sys
import os
sys.path.insert(0, os.path.abspath('..'))
from src.retriever import ComplaintRetriever
from src.prod_vector_store import build_vector_store,load_vector_store
from src.generator import ComplaintGenerator
from src.rag import ComplaintRAG



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Build Vector Store

In [2]:
collection = build_vector_store(
    parquet_path="../data/raw/complaint_embeddings.parquet",
    persist_directory="../data/vector_store",
    collection_name="complaints"
)

Loaded 1,375,327 records


Indexing Chunks:   0%|          | 0/252 [00:00<?, ?it/s]

Successfully indexed 1,375,327 chunks


# Load Vector Store

In [6]:
collection = load_vector_store(
    persist_directory="../data/vector_store",
    collection_name="complaints"
)

Loaded collection with 1,375,327 chunks


# Test Retriever

In [7]:
retriever = ComplaintRetriever(
    persist_directory="../data/vector_store"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Ask a question

In [8]:
results = retriever.retrieve(
    "Why are customers unhappy with credit cards?",
    k=5
)

In [9]:
for i, doc in enumerate(
    results["documents"][0],
    start=1
):

    print("=" * 80)
    print(f"Result {i}")
    print("=" * 80)

    print(doc[:1000])

    print("\n")

Result 1
card company and was very unhappy and frustrated. as a consumer i feel that we apply for new credit cards because of the features and benefits they offer, however we need to understand how to use them. i am not happy with the customer service and i am not happy with the misinformation i was given. i have been given misinformation by several customer service representatives and i feel that the credit card company is taking advantage of consumers who dont understand the features and benefits.


Result 2
creditors. i have an exceptional payment history. there was no reason for them to reduce my credit limit at all doing so caused me harm by making my credit usage shoot up. what the is up with these companies it like here is the card but don't use it!


Result 3
credit card companies think of their card holders. the indifferent response to a long-term card holder again came to me as a shock given how this particular company was receptive to me in the past when i called about other

In [17]:
from src.rag import ComplaintRAG

rag = ComplaintRAG()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [21]:
result = rag.ask(
    "Why are customers unhappy with credit cards?"
)

print(result["sources"])

[{'total_chunks': 4, 'company': 'BARCLAYS BANK DELAWARE', 'issue': 'Other features, terms, or problems', 'complaint_id': '3509835', 'sub_issue': 'Problem with customer service', 'chunk_index': 2, 'date_received': '2020-01-27', 'product_category': 'Credit Card', 'state': 'CA', 'product': 'Credit card or prepaid card'}, {'product_category': 'Credit Card', 'chunk_index': 1, 'product': 'Credit card', 'sub_issue': 'Other problem', 'issue': 'Other features, terms, or problems', 'state': 'MI', 'complaint_id': '8892605', 'company': 'DISCOVER BANK', 'date_received': '2024-05-01', 'total_chunks': 2}, {'date_received': '2019-11-29', 'chunk_index': 7, 'complaint_id': '3453497', 'sub_issue': 'Unexpected increase in interest rate', 'product_category': 'Credit Card', 'product': 'Credit card or prepaid card', 'total_chunks': 12, 'company': 'AMERICAN EXPRESS COMPANY', 'state': 'NY', 'issue': 'Fees or interest'}, {'state': 'CA', 'sub_issue': "Credit card company isn't resolving a dispute about a purchas

# RAG Evaluation

In [22]:
evaluation_questions = [
    "Why are customers unhappy with credit cards?",
    "What are the most common complaints about personal loans?",
    "Why are customers dissatisfied with savings accounts?",
    "What issues do customers report when using money transfer services?",
    "What problems do customers experience during credit card applications?",
    "What account management issues are frequently reported by credit card customers?",
    "Are there recurring complaints related to unexpected fees or charges?",
    "What customer service issues appear across multiple financial products?",
    "What are the main causes of disputes between customers and financial institutions?",
    "What emerging complaint trends should product managers monitor?"
]

In [23]:
results = []

for question in evaluation_questions:
    
    response = rag.ask(question)

    results.append({
        "Question": question,
        "Answer": response["answer"],
        "Sources": response["sources"][:2]  # first 2 sources
    })

print(f"Evaluated {len(results)} questions.")

Evaluated 10 questions.


In [24]:
for result in results:
    
    print("=" * 100)
    print("QUESTION:")
    print(result["Question"])

    print("\nANSWER:")
    print(result["Answer"])

    print("\nSOURCES:")
    for source in result["Sources"]:
        print(
            f"- {source['product_category']} | "
            f"{source['issue']} | "
            f"{source['company']}"
        )

    print("\n")

QUESTION:
Why are customers unhappy with credit cards?

ANSWER:
Based on the retrieved complaint excerpts, customers appear to be unhappy with credit cards due to several reasons:

1. Misinformation and poor customer service: Multiple complaints mention receiving incorrect information from customer service representatives, leading to frustration and dissatisfaction.
2. Reduction in credit limit without justification: Several customers expressed concern over having their credit limits reduced without explanation, which caused them harm by increasing their credit usage.
3. Unfair treatment and lack of loyalty: Customers feel that credit card companies do not value long-term card holders, with some experiencing indifferent or dismissive responses from customer service.
4. Unexpected interest rate increases: At least one customer mentioned an unexpected increase in their interest rate, making it more difficult to pay off their credit card debt balance.
5. General dissatisfaction with the c

In [27]:
import pandas as pd

evaluation_df = pd.DataFrame({
    "Question": [r["Question"] for r in results],
    "Generated Answer": [r["Answer"] for r in results],
    "Sources": [r["Sources"] for r in results],
    "Quality Score": [""] * len(results),
    "Comments/Analysis": [""] * len(results)
})

evaluation_df

,Question,Generated Answer,Sources,Quality Score,Comments/Analysis
0,Why are customers unhappy with credit cards?,"Based on the retrieved complaint excerpts, cus...","[{'company': 'BARCLAYS BANK DELAWARE', 'produc...",,
1,What are the most common complaints about pers...,"Based on the retrieved complaint excerpts, the...","[{'chunk_index': 1, 'issue': 'Managing an acco...",,
2,Why are customers dissatisfied with savings ac...,"Based on the retrieved complaint excerpts, cus...","[{'sub_issue': 'Banking errors', 'total_chunks...",,
3,What issues do customers report when using mon...,"Based on the provided complaint excerpts, cust...","[{'complaint_id': '2765911', 'product': 'Money...",,
4,What problems do customers experience during c...,"Based on the provided complaint excerpts, cust...","[{'state': 'WA', 'chunk_index': 3, 'complaint_...",,
5,What account management issues are frequently ...,"Based on the provided complaint excerpts, I ha...","[{'chunk_index': 4, 'complaint_id': '4276608',...",,
6,Are there recurring complaints related to unex...,"Yes, there are recurring complaints related to...","[{'product': 'Credit card or prepaid card', 't...",,
7,What customer service issues appear across mul...,"Based on the provided context, I do not have e...","[{'issue': 'Getting a credit card', 'total_chu...",,
8,What are the main causes of disputes between c...,"Based on the provided complaint excerpts, the ...","[{'total_chunks': 5, 'company': 'U.S. BANCORP'...",,
9,What emerging complaint trends should product ...,"Based on the retrieved complaint excerpts, eme...","[{'complaint_id': '3184117', 'company': 'WELLS...",,


In [28]:
evaluation_df.to_csv("../data/processed/evaluation.csv", index=False)